A edição impressa e o e-book de *Think Python 3e*, de Allen B. Downey, podem ser adquiridos na
[Bookshop.org](https://bookshop.org/a/98697/9781098155438) e na
[Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=oreilly20-20&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325).

Esta tradução educacional em português brasileiro é gratuita, sem fins lucrativos, e destina-se a quem quer aprender lógica de programação em Python.

# Análise e geração de texto

Neste ponto, cobrimos as estruturas de dados centrais de Python, listas, dicionários e tuplas, e alguns algoritmos que as usam.
Neste capítulo, vamos usá-las para explorar análise de texto e geração de Markov:

* Análise de texto é um modo de descrever as relações estatísticas entre as palavras de um documento, como a probabilidade de uma palavra ser seguida por outra, e

* Geração de Markov é um modo de gerar texto novo com palavras e frases semelhantes às do texto original.

Esses algoritmos se parecem com partes de um grande modelo de linguagem (LLM), que é o componente-chave de um chatbot.

Vamos começar contando quantas vezes cada palavra aparece em um livro.
Depois vamos olhar pares de palavras e montar uma lista das palavras que podem seguir cada palavra.
Vamos fazer uma versão simples de um gerador de Markov e, como exercício, você terá a chance de fazer uma versão mais geral.

## Palavras únicas

Como primeiro passo rumo à análise de texto, vamos ler um livro, *The Strange Case Of Dr. Jekyll And Mr. Hyde*, de Robert Louis Stevenson, e contar o número de palavras únicas.
As instruções para baixar o livro estão no notebook deste capítulo.

A célula a seguir baixa o livro do Project Gutenberg.

A versão disponível no Project Gutenberg inclui informações sobre o livro no início e informações de licença no fim.
Vamos usar `clean_file` do Capítulo 8 para remover esse material e gravar um arquivo "limpo" que contém só o texto do livro.

Vamos usar um laço `for` para ler linhas do arquivo e `split` para dividir as linhas em palavras.
Depois, para acompanhar as palavras únicas, vamos guardar cada palavra como chave em um dicionário.

O comprimento do dicionário é o número de palavras únicas: cerca de `6000` por essa forma de contar.
Mas se as inspecionarmos, veremos que algumas não são palavras válidas.

Por exemplo, vamos olhar as palavras mais longas em `unique_words`.
Podemos usar `sorted` para ordenar as palavras, passando a função `len` como argumento nomeado para que as palavras sejam ordenadas por comprimento.

O índice de fatiamento, `[-5:]`, seleciona os últimos `5` elementos da lista ordenada, que são as palavras mais longas. 

A lista inclui algumas palavras legitimamente longas, como "circumscription", e algumas palavras hifenizadas, como "chocolate-coloured".
Mas algumas das "palavras" mais longas são, na verdade, duas palavras separadas por um travessão.
E outras palavras incluem pontuação como pontos, pontos de exclamação e aspas.

Então, antes de seguir, vamos lidar com travessões e outra pontuação.

## Pontuação

Para identificar as palavras no texto, precisamos lidar com duas questões:

* Quando um travessão aparece em uma linha, devemos substituí-lo por um espaço: assim, quando usarmos `split`, as palavras ficarão separadas.

* Depois de separar as palavras, podemos usar `strip` para remover a pontuação.

Para tratar a primeira questão, podemos usar a função a seguir, que recebe uma string, substitui travessões por espaços, divide a string e devolve a lista resultante.

Observe que `split_line` só substitui travessões, não hífens.
Aqui está um exemplo.

Agora, para remover pontuação do início e do fim de cada palavra, podemos usar `strip`, mas precisamos de uma lista de caracteres considerados pontuação.

Os caracteres nas strings de Python estão em Unicode, um padrão internacional usado para representar letras em quase todos os alfabetos, números, símbolos, sinais de pontuação e mais.
O módulo `unicodedata` oferece uma função `category` que podemos usar para saber quais caracteres são pontuação.
Dada uma letra, ela devolve uma string com informações sobre a categoria da letra.

A string de categoria de `'A'` é `'Lu'`: o `'L'` significa que é uma letra e o `'u'` significa que é maiúscula.

A string de categoria de `'.'` é `'Po'`: o `'P'` significa que é pontuação e o `'o'` significa que a subcategoria é "other".

Podemos encontrar os sinais de pontuação no livro verificando caracteres cujas categorias começam com `'P'`.
O laço a seguir guarda os sinais de pontuação únicos em um dicionário.

Para montar uma lista de sinais de pontuação, podemos juntar as chaves do dicionário em uma string.

Agora que sabemos quais caracteres do livro são pontuação, podemos escrever uma função que recebe uma palavra, remove a pontuação do início e do fim e a converte para minúsculas.

Aqui está um exemplo.

Como `strip` remove caracteres do início e do fim, deixa as palavras hifenizadas em paz.

Agora aqui está um laço que usa `split_line` e `clean_word` para identificar as palavras únicas do livro.

Com essa definição mais rigorosa do que é uma palavra, há cerca de 4000 palavras únicas.
E podemos confirmar que a lista das palavras mais longas foi limpa.

Agora vamos ver quantas vezes cada palavra é usada.

## Frequências de palavras

O laço a seguir calcula a frequência de cada palavra única.

Na primeira vez em que vemos uma palavra, inicializamos a frequência em `1`. Se virmos a mesma palavra de novo mais tarde, incrementamos a frequência.

Para ver quais palavras aparecem com mais frequência, podemos usar `items` para obter os pares chave-valor de `word_counter` e ordená-los pelo segundo elemento do par, que é a frequência.
Primeiro vamos definir uma função que seleciona o segundo elemento.

Agora podemos usar `sorted` com dois argumentos nomeados:

* `key=second_element` significa que os itens serão ordenados segundo as frequências das palavras.

* `reverse=True` significa que os itens serão ordenados em ordem inversa, com as palavras mais frequentes primeiro.

Aqui estão as cinco palavras mais frequentes.

Na próxima seção, vamos encapsular esse laço em uma função.
E vamos usá-la para demonstrar um recurso novo: parâmetros opcionais.

## Parâmetros opcionais

Já usamos funções nativas que recebem parâmetros opcionais.
Por exemplo, `round` recebe um parâmetro opcional chamado `ndigits` que indica quantas casas decimais manter.

Mas não são só as funções nativas: nós também podemos escrever funções com parâmetros opcionais.
Por exemplo, a função a seguir recebe dois parâmetros, `word_counter` e `num`.

O segundo parâmetro parece uma instrução de atribuição, mas não é: é um parâmetro opcional.

Se você chamar esta função com um argumento, `num` recebe o **valor padrão**, que é `5`.

Se você chamar esta função com dois argumentos, o segundo argumento é atribuído a `num` no lugar do valor padrão.

Nesse caso, diríamos que o argumento opcional **sobrescreve** o valor padrão.

Se uma função tem parâmetros obrigatórios e opcionais, todos os obrigatórios precisam vir primeiro, seguidos pelos opcionais.

## Subtração de dicionários

Suponha que queremos revisar a ortografia de um livro, isto é, encontrar uma lista de palavras que podem estar escritas errado.
Uma forma de fazer isso é encontrar palavras do livro que não aparecem em uma lista de palavras válidas.
Em capítulos anteriores, usamos uma lista de palavras consideradas válidas em jogos como o Scrabble.
Agora vamos usar essa lista para revisar a ortografia de Robert Louis Stevenson.

Podemos pensar nesse problema como subtração de conjuntos: queremos encontrar todas as palavras de um conjunto (as palavras do livro) que não estão no outro (as palavras da lista).

A célula a seguir baixa a lista de palavras.

Como já fizemos, podemos ler o conteúdo de `words.txt` e dividi-lo em uma lista de strings.

Em seguida, vamos guardar as palavras como chaves de um dicionário, para usar o operador `in` e verificar depressa se uma palavra é válida.

Agora, para identificar palavras que aparecem no livro mas não na lista de palavras, vamos usar `subtract`, que recebe dois dicionários como parâmetros e devolve um dicionário novo que contém todas as chaves de um que não estão no outro.

Eis como a usamos.

Para obter uma amostra de palavras que podem estar escritas errado, podemos imprimir as palavras mais comuns em `diff`.

As "palavras erradas" mais comuns são sobretudo nomes e algumas palavras de uma letra (Mr. Utterson é amigo e advogado do Dr. Jekyll).

Se selecionarmos palavras que só aparecem uma vez, é mais provável que sejam erros reais.
Podemos fazer isso percorrendo os itens e montando uma lista de palavras com frequência `1`.

Aqui estão os últimos elementos da lista.

A maior parte delas são palavras válidas que não estão na lista de palavras.
Mas `'reindue'` parece um erro de `'reinduce'`, então pelo menos encontramos um erro legítimo.

## Números aleatórios

Como passo rumo à geração de texto de Markov, em seguida vamos escolher uma sequência aleatória de palavras de `word_counter`.
Mas primeiro vamos falar de aleatoriedade.

Dadas as mesmas entradas, a maior parte dos programas de computador é **determinística**, o que significa que gera as mesmas saídas todas as vezes.
O determinismo em geral é uma coisa boa, já que esperamos que o mesmo cálculo produza o mesmo resultado.
Em algumas aplicações, porém, queremos que o computador seja imprevisível.
Jogos são um exemplo, mas há mais.

Tornar um programa verdadeiramente não determinístico acaba sendo difícil, mas há formas de fingir.
Uma delas é usar algoritmos que geram números **pseudoaleatórios**.
Números pseudoaleatórios não são verdadeiramente aleatórios porque são gerados por um cálculo determinístico, mas só de olhar os números é praticamente impossível distingui-los de aleatórios.

O módulo `random` oferece funções que geram números pseudoaleatórios, que daqui para frente vou chamar simplesmente de "aleatórios".
Podemos importá-lo assim.

O módulo `random` oferece uma função chamada `choice` que escolhe um elemento de uma lista ao acaso, com cada elemento tendo a mesma probabilidade de ser escolhido.

Se você chamar a função de novo, pode obter o mesmo elemento de novo, ou um diferente.

No longo prazo, esperamos obter cada elemento aproximadamente o mesmo número de vezes.

Se você usa `choice` com um dicionário, recebe um `KeyError`.

Para escolher uma chave aleatória, você precisa colocar as chaves em uma lista e depois chamar `choice`.

Se gerarmos uma sequência aleatória de palavras, ela não faz muito sentido.

Parte do problema é que não estamos levando em conta que algumas palavras são mais comuns do que outras.
Os resultados serão melhores se escolhermos palavras com "pesos" diferentes, de modo que algumas sejam escolhidas com mais frequência do que outras.

Se usarmos os valores de `word_counter` como pesos, cada palavra é escolhida com uma probabilidade que depende da frequência.

O módulo `random` oferece outra função chamada `choices` que recebe pesos como argumento opcional.

E recebe outro argumento opcional, `k`, que especifica o número de palavras a selecionar.

O resultado é uma lista de strings que podemos juntar em algo que se parece mais com uma frase.

Se você escolhe palavras do livro ao acaso, tem uma ideia do vocabulário, mas uma série de palavras aleatórias raramente faz sentido porque não há relação entre palavras sucessivas.
Por exemplo, em uma frase real você espera que um artigo como "the" seja seguido por um adjetivo ou um substantivo, e provavelmente não por um verbo ou advérbio.
Então o próximo passo é olhar essas relações entre palavras.

## Bigramas

Em vez de olhar uma palavra de cada vez, agora vamos olhar sequências de duas palavras, chamadas **bigramas**.
Uma sequência de três palavras se chama **trigrama**, e uma sequência com um número não especificado de palavras se chama **n-grama**.

Vamos escrever um programa que encontra todos os bigramas do livro e o número de vezes em que cada um aparece.
Para guardar os resultados, vamos usar um dicionário em que

* As chaves são tuplas de strings que representam bigramas, e 

* Os valores são inteiros que representam frequências.

Vamos chamá-lo de `bigram_counter`.

A função a seguir recebe uma lista de duas strings como parâmetro.
Primeiro ela monta uma tupla das duas strings, que pode ser usada como chave em um dicionário.
Depois adiciona a chave a `bigram_counter`, se ela não existir, ou incrementa a frequência se existir.

À medida que percorremos o livro, precisamos acompanhar cada par de palavras consecutivas.
Assim, se virmos a sequência "man is not truly one", adicionaríamos os bigramas "man is", "is not", "not truly", e assim por diante.

Para acompanhar esses bigramas, vamos usar uma lista chamada `window`, porque é como uma janela que desliza pelas páginas do livro, mostrando só duas palavras de cada vez.
Inicialmente, `window` está vazia.

Vamos usar a função a seguir para processar as palavras uma de cada vez.

Na primeira vez em que esta função é chamada, ela acrescenta a palavra dada a `window`.
Como há só uma palavra na janela, ainda não temos um bigrama, então a função termina.

Na segunda vez em que é chamada, e em todas as vezes seguintes, ela acrescenta uma segunda palavra a `window`.
Como há duas palavras na janela, chama `count_bigram` para acompanhar quantas vezes cada bigrama aparece.
Depois usa `pop` para remover a primeira palavra da janela.

O programa a seguir percorre as palavras do livro e as processa uma de cada vez.

O resultado é um dicionário que mapeia cada bigrama para o número de vezes em que ele aparece.
Podemos usar `print_most_common` para ver os bigramas mais comuns.

Olhando esses resultados, temos uma ideia de quais pares de palavras têm mais chance de aparecer juntos.
Também podemos usar os resultados para gerar texto aleatório, assim.

`bigrams` é uma lista dos bigramas que aparecem no livro.
`weights` é uma lista das frequências deles, então `random_bigrams` é uma amostra em que a probabilidade de um bigrama ser selecionado é proporcional à frequência. 

Aqui estão os resultados.

Esse jeito de gerar texto é melhor do que escolher palavras aleatórias, mas ainda não faz muito sentido.

## Análise de Markov

Podemos fazer melhor com a análise de texto por cadeia de Markov, que calcula, para cada palavra de um texto, a lista de palavras que vêm a seguir.
Como exemplo, vamos analisar estas letras da canção *Eric, the Half a Bee*, do Monty Python:

Para guardar os resultados, vamos usar um dicionário que mapeia cada palavra para a lista de palavras que a seguem.

Como exemplo, vamos começar com as duas primeiras palavras da canção.

Se a primeira palavra não está em `successor_map`, precisamos adicionar um item novo que mapeia da primeira palavra para uma lista contendo a segunda palavra.

Se a primeira palavra já está no dicionário, podemos procurá-la para obter a lista de sucessores que já vimos e acrescentar o novo.

A função a seguir encapsula esses passos.

Se o mesmo bigrama aparecer mais de uma vez, a segunda palavra é adicionada à lista mais de uma vez.
Desse modo, `successor_map` acompanha quantas vezes cada sucessor aparece.

Como fizemos na seção anterior, vamos usar uma lista chamada `window` para guardar pares de palavras consecutivas.
E vamos usar a função a seguir para processar as palavras uma de cada vez.

Eis como a usamos para processar as palavras da canção.

E aqui estão os resultados.

A palavra `'half'` pode ser seguida por `'a'`, `'not'` ou `'the'`.
A palavra `'a'` pode ser seguida por `'bee'` ou `'vis'`.
A maior parte das outras palavras aparece só uma vez, então é seguida por uma única palavra.

Agora vamos analisar o livro.

Podemos procurar qualquer palavra e encontrar as palavras que podem segui-la.

Nesta lista de sucessores, observe que a palavra `'to'` aparece três vezes: os outros sucessores só aparecem uma vez.

## Gerando texto

Podemos usar os resultados da seção anterior para gerar texto novo com as mesmas relações entre palavras consecutivas que no original.
Eis como funciona:

* Começando com qualquer palavra que aparece no texto, procuramos seus sucessores possíveis e escolhemos um ao acaso.

* Depois, usando a palavra escolhida, procuramos seus sucessores possíveis e escolhemos um ao acaso.

Podemos repetir esse processo para gerar quantas palavras quisermos.
Como exemplo, vamos começar com a palavra `'although'`.
Aqui estão as palavras que podem segui-la.

Podemos usar `choice` para escolher da lista com probabilidade igual.

Se a mesma palavra aparecer mais de uma vez na lista, ela tem mais chance de ser selecionada.

Repetindo esses passos, podemos usar o laço a seguir para gerar uma série mais longa.

O resultado soa mais como uma frase real, mas ainda não faz muito sentido.

Podemos fazer melhor usando mais de uma palavra como chave em `successor_map`.
Por exemplo, podemos criar um dicionário que mapeia cada bigrama, ou trigrama, para a lista de palavras que vêm a seguir.
Como exercício, você terá a chance de implementar essa análise e ver como ficam os resultados.

## Depuração

Neste ponto estamos escrevendo programas mais substanciais, e você pode descobrir que está gastando mais tempo depurando.
Se estiver travado em um bug difícil, aqui vão algumas coisas para tentar:

* Ler: examine o código, leia-o em voz alta e confira se ele diz o que você quis dizer.

* Executar: experimente fazendo mudanças e rodando versões diferentes. Muitas vezes, se você exibir a coisa certa no lugar certo do programa, o problema fica óbvio, mas às vezes é preciso construir andaimes.

* Ruminar: reserve um tempo para pensar! Que tipo de erro é: de sintaxe, de execução
    ou semântico? Que informação você pode obter das mensagens de erro,
    ou da saída do programa? Que tipo de erro poderia causar
    o problema que você está vendo? O que você mudou por último, antes de o
    problema aparecer?

* Rubberducking: se você explica o problema para outra pessoa, às vezes encontra a
    resposta antes de terminar de fazer a pergunta. Muitas vezes você não precisa
    da outra pessoa; poderia só falar com um patinho de borracha. E essa é
    a origem da estratégia bem conhecida chamada **depuração do patinho
    de borracha**. Não estou inventando: veja
    <https://en.wikipedia.org/wiki/Rubber_duck_debugging>.

* Recuar: em algum momento, o melhor a fazer é voltar atrás, desfazendo mudanças
    recentes, até chegar a um programa que funciona. Depois você pode começar a reconstruir.
    
* Descansar: se você der uma pausa ao cérebro, às vezes ele encontra o problema por você.

Quem está começando às vezes fica preso em uma dessas atividades e esquece as outras. Cada atividade tem o próprio modo de falha.

Por exemplo, ler o código funciona se o problema é um erro de digitação, mas não se o problema é um mal-entendido conceitual.
Se você não entende o que o programa faz, pode lê-lo 100 vezes e nunca ver o erro, porque o erro está na sua cabeça.

Rodar experimentos pode funcionar, sobretudo se você rodar testes pequenos e simples.
Mas se você roda experimentos sem pensar nem ler o código, pode levar muito tempo para descobrir o que está acontecendo.

Você precisa reservar tempo para pensar. Depurar é como uma ciência experimental. Você deveria ter pelo menos uma hipótese sobre qual é o problema. Se houver duas ou mais possibilidades, tente pensar em um teste que eliminaria uma delas.

Mas mesmo as melhores técnicas de depuração falham se houver erros demais,
ou se o código que você está tentando consertar for grande e complicado demais.
Às vezes a melhor opção é recuar, simplificando o programa até
voltar a algo que funciona.

Quem está começando muitas vezes reluta em recuar porque não
aguenta apagar uma linha de código (mesmo que esteja errada). Se isso fizer você
se sentir melhor, copie o programa para outro arquivo antes de começar
a enxugá-lo. Depois você pode copiar os pedaços de volta um de cada vez.

Encontrar um bug difícil exige ler, executar, ruminar, recuar e, às vezes, descansar.
Se você ficar preso em uma dessas atividades, tente as outras.

## Glossário

**valor padrão:**
O valor atribuído a um parâmetro se nenhum argumento for fornecido.

**sobrescrever:**
 Substituir um valor padrão por um argumento.

**determinístico:**
 Um programa determinístico faz a mesma coisa cada vez que roda, dadas as mesmas entradas.

**pseudoaleatório:**
 Uma sequência pseudoaleatória de números parece aleatória, mas é gerada por um programa determinístico.

**bigrama:**
Uma sequência de dois elementos, muitas vezes palavras.

**trigrama:**
Uma sequência de três elementos.

**n-grama:**
Uma sequência de um número não especificado de elementos.

**depuração do patinho de borracha:**
Um modo de depurar explicando um problema em voz alta para um objeto inanimado.

## Exercícios

### Peça a um assistente virtual

Em `add_bigram`, a instrução `if` cria uma lista nova ou acrescenta um elemento a uma lista existente, conforme a chave já esteja ou não no dicionário.

Os dicionários oferecem um método chamado `setdefault` que faz a mesma coisa de forma mais concisa.
Pergunte a um assistente virtual como ele funciona, ou copie `add_word` para um assistente e pergunte: "Can you rewrite this using `setdefault`?"

Neste capítulo implementamos análise e geração de texto por cadeia de Markov.
Se tiver curiosidade, pode pedir a um assistente virtual mais informações sobre o tema.
Uma das coisas que você pode aprender é que os assistentes virtuais usam algoritmos semelhantes em vários aspectos, e diferentes em outros, igualmente importantes.
Pergunte a um assistente: "What are the differences between large language models like GPT and Markov chain text analysis?"

### Exercício

Escreva uma função que conte o número de vezes em que cada trigrama (sequência de três palavras) aparece. 
Se você testar a função com o texto de *Dr. Jekyll and Mr. Hyde*, deve descobrir que o trigrama mais comum é "said the lawyer".

Dica: escreva uma função chamada `count_trigram` semelhante a `count_bigram`. Depois escreva uma função chamada `process_word_trigram` semelhante a `process_word_bigram`.

Você pode usar o laço a seguir para ler o livro e processar as palavras.

Depois use `print_most_common` para encontrar os trigramas mais comuns do livro.

### Exercício

Agora vamos implementar análise de texto por cadeia de Markov com um mapeamento de cada bigrama para uma lista de sucessores possíveis.

Partindo de `add_bigram`, escreva uma função chamada `add_trigram` que receba uma lista de três palavras e adicione ou atualize um item em `successor_map`, usando as duas primeiras palavras como chave e a terceira palavra como sucessor possível.

Aqui está uma versão de `process_word_trigram` que chama `add_trigram`.

Você pode usar o laço a seguir para testar a função com as letras de "Eric, the Half a Bee".

Se a função funcionar como pretendido, o predecessor `('half', 'a')` deve mapear para uma lista com o único elemento `'bee'`.
De fato, como acontece, cada bigrama desta canção aparece só uma vez, então todos os valores em `successor_map` têm um único elemento.

Você pode usar o laço a seguir para testar a função com as palavras do livro.

No próximo exercício, você usará os resultados para gerar texto aleatório novo.

### Exercício

Para este exercício, vamos supor que `successor_map` é um dicionário que mapeia cada bigrama para a lista de palavras que o seguem.

Para gerar texto aleatório, vamos começar escolhendo uma chave aleatória de `successor_map`.

Agora escreva um laço que gere mais 50 palavras seguindo estes passos:

1. Em `successor_map`, procure a lista de palavras que podem seguir `bigram`.

2. Escolha uma delas ao acaso e imprima-a.

3. Para a iteração seguinte, monte um bigrama novo que contenha a segunda palavra de `bigram` e o sucessor escolhido.

Por exemplo, se começarmos com o bigrama `('doubted', 'if')` e escolhermos `'from'` como sucessor, o próximo bigrama é `('if', 'from')`.

Se tudo estiver funcionando, você deve perceber que o texto gerado é reconhecivelmente semelhante em estilo ao original, e algumas frases fazem sentido, mas o texto pode vaguear de um assunto para outro.

Como exercício bônus, modifique a solução dos dois últimos exercícios para usar trigramas como chaves em `successor_map` e veja que efeito isso tem nos resultados.

[Pense em Python: 3ª edição](https://allendowney.github.io/ThinkPython/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

Tradução educacional para o português brasileiro, sem fins lucrativos.

Licença do código: [Licença MIT](https://mit-license.org/)

Licença do texto: [Creative Commons Atribuição-NãoComercial-CompartilhaIgual 4.0 Internacional](https://creativecommons.org/licenses/by-nc-sa/4.0/)